In [1]:
import numpy as np
import pandas as pd
import os, sys, datetime, time, glob, random
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from scipy.stats import pearsonr
from bayes_opt import BayesianOptimization, UtilityFunction
import pyreadr
from patsy import dmatrix
from statsmodels.graphics.api import abline_plot
import torch as tch

# Configurar PyTorch
GPU = True if tch.cuda.is_available() else False
Num_Threads = 1
os.environ["OMP_NUM_THREADS"] = str(Num_Threads)
os.environ["MKL_NUM_THREADS"] = str(Num_Threads)
tch.set_num_threads(Num_Threads)
tch.set_num_interop_threads(Num_Threads)

print("Torch GPU disponible:", GPU)


ImportError: cannot import name 'UtilityFunction' from 'bayes_opt' (C:\Users\LENOVO\anaconda3\Lib\site-packages\bayes_opt\__init__.py)

In [ ]:
# Asegúrate de tener Funs_Auxs.py y MLP_v8_0.py en el mismo directorio
Dir = os.getcwd()
sys.path.append(Dir)

from Funs_Auxs import *
from MLP_v8_0 import *


In [ ]:
Dir_DataSet = os.path.normpath(os.path.join(Dir, 'dat-PP'))
Folds_df = pd.read_csv(os.path.join(Dir_DataSet, '10Folds_df.csv'))
DataSets = [f'Wheat_{j+1}_AL' for j in range(6)]

print("Datasets disponibles:", DataSets)


In [ ]:
def ejecutar_pipeline(dn=1, trait="GY"):
    DataNumber = dn - 1
    Dir_dat = os.path.join(Dir_DataSet, DataSets[DataNumber] + '.RData')
    OD = pyreadr.read_r(Dir_dat)

    Pheno = OD['Pheno'].rename(columns={'Line': 'GID'})
    y = Pheno[trait].to_numpy(dtype=float).reshape(-1, 1)

    XM = OD['Markers'].apply(np.round)
    Scaler_Markers = StandardScaler()
    XM = Scaler_Markers.fit_transform(XM)
    G = np.matmul(XM, XM.T) / XM.shape[1]
    G = pd.DataFrame(G)
    G5 = XR2_f(G)

    Order = OD['Markers'].index
    Pheno['GID'] = pd.Categorical(Pheno['GID'], categories=Order, ordered=True)
    ZL = pd.get_dummies(Pheno['GID'])
    XL = np.matmul(ZL.values, G5.loc[Match_f(ZL.columns, Order)].values)
    x_ls_all = [XL]

    GIDs = Pheno['GID'].unique()
    Fold_df = Folds_df[Folds_df['DataSet'] == DataSets[DataNumber]]
    K = Fold_df['Fold'].nunique()

    Hypers_Fixed = {
        'Epochs': 128, 'Batch_size': len(y), 'Seed': 42, 'Iters': 250,
        'IK': 10, 'Use': 10,
        'Device': torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    }

    print("Pipeline iniciado para:", DataSets[DataNumber])


In [ ]:
# Ejecutar la función para el primer dataset
ejecutar_pipeline(dn=1)
